# Part C: PPO Hospital Environment

Part C is now standard single-agent PPO rather than MAPPO because one controller makes the hospital-flow decision at each timestep. This removes PettingZoo-style multi-agent dictionaries and uses a Gymnasium-style `reset` / `step` API.


## Imports

These imports support queues, patient records, NumPy observations, and Gymnasium-style action/observation spaces. No PPO training or neural networks are created here.


In [11]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from typing import Any, Deque, Dict, Iterable, List, Optional, Tuple

import numpy as np

try:
    from gymnasium import spaces
except ImportError:
    class _Discrete:
        def __init__(self, n: int):
            self.n = int(n)

        def sample(self) -> int:
            return int(np.random.randint(self.n))

    class _Box:
        def __init__(self, low, high, shape=None, dtype=np.float32):
            self.low = np.asarray(low, dtype=dtype)
            self.high = np.asarray(high, dtype=dtype)
            self.shape = shape or self.low.shape
            self.dtype = dtype

    class _Spaces:
        Discrete = _Discrete
        Box = _Box

    spaces = _Spaces()


## CUDA Requirement

This notebook is configured to stop immediately unless PyTorch imports successfully and detects a CUDA GPU. The environment itself is NumPy-based, but later PPO training is required to run on CUDA.


In [12]:
REQUIRE_CUDA = True

try:
    import torch
except Exception as exc:
    raise RuntimeError(
        "CUDA is required, but PyTorch could not be imported. "
        "Install a working CUDA-enabled PyTorch build before running this notebook."
    ) from exc

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required for this notebook, but PyTorch did not detect a CUDA GPU. "
        "Stop execution and fix the CUDA/PyTorch installation before continuing."
    )

DEVICE = torch.device("cuda")
print(f"Using CUDA: {torch.cuda.get_device_name(0)}")


Using CUDA: NVIDIA GeForce RTX 3060 Laptop GPU


## Environment Configuration and Constants

Constants define staff schedules, action meanings, patient stages/types, shift encoding, and the lightweight patient record used by the environment.


In [13]:
STAFF_SCHEDULE = {
    "day": {
        "nurses": 6,
        "doctors": 5,
        "physicians": 3,
        "radiologists": 2,
        "receptionists": 3,
        "administrators": 2,
        "paramedics": 4,
    },
    "evening": {
        "nurses": 4,
        "doctors": 3,
        "physicians": 2,
        "radiologists": 1,
        "receptionists": 2,
        "administrators": 1,
        "paramedics": 3,
    },
    "night": {
        "nurses": 2,
        "doctors": 1,
        "physicians": 1,
        "radiologists": 1,
        "receptionists": 1,
        "administrators": 1,
        "paramedics": 2,
    },
}

PATIENT_TYPES = {
    0: "emergency",
    1: "urgent",
    2: "elective_minor",
}

PATIENT_STAGES = {
    0: "arrival",
    1: "registration",
    2: "assessment",
    3: "triage",
    4: "imaging_tests",
    5: "treatment",
    6: "resus",
    7: "major",
    8: "minor",
    9: "general_admission",
    10: "icu_admission",
    11: "discharge",
    12: "delay_wait",
}

ACTION_MEANINGS = {
    0: "do_nothing",
    1: "register_patient",
    2: "move_to_assessment",
    3: "send_to_minor",
    4: "send_to_major",
    5: "send_to_resus",
    6: "request_imaging",
    7: "process_imaging",
    8: "treat_patient",
    9: "admit_general",
    10: "admit_icu",
    11: "discharge_patient",
    12: "delay_patient",
    13: "preserve_icu_capacity",
    14: "prioritise_emergency_flow",
}

SHIFT_ENCODING = {
    "day": 0,
    "evening": 1,
    "night": 2,
}

PATIENT_TYPE_PROBABILITIES = np.asarray([0.2, 0.3, 0.5], dtype=np.float64)

@dataclass(eq=False)
class Patient:
    patient_type: int
    stage: int = 0
    waiting_time: int = 0
    needs_imaging: bool = False
    area: Optional[str] = None


## `HospitalPPOEnv` Class

This is a single-agent operational hospital-flow environment. The action is one discrete hospital decision, and the reward balances responsiveness, waiting time, congestion, admissions, discharge, and ICU preservation.


In [14]:
class HospitalPPOEnv:
    """Single-agent operational hospital-flow environment for later PPO training.

    This is not a grid-world. It models queues, stages, staff capacity, beds,
    admissions, discharges, waiting time, and congestion.
    """

    metadata = {"name": "HospitalPPOEnv"}

    def __init__(
        self,
        registration_queue_max: int = 20,
        assessment_queue_max: int = 20,
        resus_max: int = 3,
        major_max: int = 8,
        minor_max: int = 10,
        general_max: int = 20,
        icu_max: int = 6,
        max_steps: int = 500,
        arrival_probability: float = 0.65,
        patient_type_probabilities: np.ndarray = PATIENT_TYPE_PROBABILITIES,
        staff_schedule: Optional[Dict[str, Dict[str, int]]] = None,
    ) -> None:
        self.registration_queue_max = int(registration_queue_max)
        self.assessment_queue_max = int(assessment_queue_max)
        self.resus_max = int(resus_max)
        self.major_max = int(major_max)
        self.minor_max = int(minor_max)
        self.general_max = int(general_max)
        self.icu_max = int(icu_max)
        self.max_steps = int(max_steps)
        self.arrival_probability = float(arrival_probability)
        self.patient_type_probabilities = np.asarray(patient_type_probabilities, dtype=np.float64)
        self.patient_type_probabilities /= self.patient_type_probabilities.sum()
        self.staff_schedule = staff_schedule or STAFF_SCHEDULE

        self.resus_release_probability = 0.25
        self.major_release_probability = 0.30
        self.minor_release_probability = 0.50
        self.general_discharge_probability = 0.35
        self.icu_discharge_probability = 0.20

        self.action_space = spaces.Discrete(15)
        self.observation_space = spaces.Box(
            low=np.zeros(18, dtype=np.float32),
            high=np.asarray(
                [
                    self.registration_queue_max,
                    self.assessment_queue_max,
                    self.resus_max,
                    self.major_max,
                    self.minor_max,
                    self.general_max,
                    self.icu_max,
                    2,
                    12,
                    self.max_steps,
                    max(v["nurses"] for v in self.staff_schedule.values()),
                    max(v["doctors"] for v in self.staff_schedule.values()),
                    max(v["physicians"] for v in self.staff_schedule.values()),
                    max(v["radiologists"] for v in self.staff_schedule.values()),
                    max(v["receptionists"] for v in self.staff_schedule.values()),
                    max(v["administrators"] for v in self.staff_schedule.values()),
                    max(v["paramedics"] for v in self.staff_schedule.values()),
                    2,
                ],
                dtype=np.float32,
            ),
            dtype=np.float32,
        )

        self._np_random = np.random.default_rng()
        self._arrival_queue: Deque[Patient] = deque()
        self._registration_patients: Deque[Patient] = deque()
        self._assessment_patients: Deque[Patient] = deque()
        self._imaging_patients: Deque[Patient] = deque()
        self._treatment_patients: Deque[Patient] = deque()
        self._admission_patients: Deque[Patient] = deque()
        self._resus_patients: Deque[Patient] = deque()
        self._major_patients: Deque[Patient] = deque()
        self._minor_patients: Deque[Patient] = deque()
        self._general_patients: Deque[Patient] = deque()
        self._icu_patients: Deque[Patient] = deque()

        self._reset_public_state()

    def reset(self, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
        if seed is not None:
            self._np_random = np.random.default_rng(seed)

        options = options or {}
        self.timestep = 0
        self._preserve_icu_mode = False

        for queue in self._all_queues():
            queue.clear()

        self.resus_free = self.resus_max
        self.major_free = self.major_max
        self.minor_free = self.minor_max
        self.general_free = self.general_max
        self.icu_free = self.icu_max
        self.shift = self._shift_from_timestep(self.timestep)
        self._apply_shift_staffing()

        initial_patients = int(options.get("initial_patients", 1))
        for _ in range(max(initial_patients, 0)):
            self._arrival_queue.append(self._new_patient())

        self._sync_public_state()
        return self._get_observation(), self._get_info(events=["reset"])

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict[str, Any]]:
        action = int(np.clip(action, 0, self.action_space.n - 1))
        self.timestep += 1
        self.shift = self._shift_from_timestep(self.timestep)
        self._apply_shift_staffing()

        reward_components = []
        events = []

        release_reward = self._release_capacity(events)
        reward_components.append(("release_throughput", release_reward))

        if self._np_random.random() < self.arrival_probability:
            self._arrival_queue.append(self._new_patient())
            events.append("new_patient_arrived")

        action_reward = self._apply_action(action, events)
        reward_components.append(("action_reward", action_reward))

        waiting_penalty = self._update_waiting_times()
        reward_components.append(("waiting_penalty", -waiting_penalty))

        congestion_penalty = self._queue_congestion_penalty()
        reward_components.append(("queue_congestion_penalty", -congestion_penalty))

        icu_reward = self._icu_preservation_reward()
        reward_components.append(("icu_preservation_reward", icu_reward))

        self._sync_public_state()
        reward = float(sum(value for _, value in reward_components))
        terminated = False
        truncated = self.timestep >= self.max_steps
        info = self._get_info(
            action=action,
            action_meaning=ACTION_MEANINGS[action],
            events=events,
            reward_components=dict(reward_components),
        )
        return self._get_observation(), reward, terminated, truncated, info

    def _apply_action(self, action: int, events: List[str]) -> float:
        if action == 0:
            events.append("do_nothing")
            return -0.1 if self._has_waiting_patients() else 0.0
        if action == 1:
            return self._register_patient(events)
        if action == 2:
            return self._move_to_assessment(events)
        if action == 3:
            return self._send_to_area("minor", events)
        if action == 4:
            return self._send_to_area("major", events)
        if action == 5:
            return self._send_to_area("resus", events)
        if action == 6:
            return self._request_imaging(events)
        if action == 7:
            return self._process_imaging(events)
        if action == 8:
            return self._treat_patient(events)
        if action == 9:
            return self._admit_general(events)
        if action == 10:
            return self._admit_icu(events)
        if action == 11:
            return self._discharge_patient(events)
        if action == 12:
            return self._delay_patient(events)
        if action == 13:
            return self._preserve_icu_capacity(events)
        if action == 14:
            return self._prioritise_emergency_flow(events)
        return 0.0

    def _register_patient(self, events: List[str]) -> float:
        if not self._arrival_queue or len(self._registration_patients) >= self.registration_queue_max:
            events.append("registration_blocked")
            return -1.0
        if self.receptionists_available <= 0:
            events.append("no_receptionist_available")
            return -1.0
        patient = self._pop_priority(self._arrival_queue)
        patient.stage = 1
        self._registration_patients.append(patient)
        events.append("patient_registered")
        return 1.0 if patient.patient_type != 0 else 2.0

    def _move_to_assessment(self, events: List[str]) -> float:
        if not self._registration_patients or len(self._assessment_patients) >= self.assessment_queue_max:
            events.append("assessment_move_blocked")
            return -1.0
        patient = self._pop_priority(self._registration_patients)
        patient.stage = 2
        self._assessment_patients.append(patient)
        events.append("patient_moved_to_assessment")
        return 1.0 if patient.patient_type != 0 else 2.0

    def _send_to_area(self, area: str, events: List[str]) -> float:
        if not self._assessment_patients:
            events.append(f"send_to_{area}_blocked_no_patient")
            return -1.0
        if self.nurses_available <= 0:
            events.append("no_nurse_available")
            return -1.0
        if getattr(self, f"{area}_free") <= 0:
            events.append(f"send_to_{area}_blocked_no_capacity")
            patient = self._peek_priority(self._assessment_patients)
            if patient and patient.patient_type == 0 and area in {"resus", "major"}:
                return -8.0
            return -2.0

        patient = self._pop_priority(self._assessment_patients)
        patient.stage = {"resus": 6, "major": 7, "minor": 8}[area]
        patient.area = area
        getattr(self, f"_{area}_patients").append(patient)
        self._treatment_patients.append(patient)
        setattr(self, f"{area}_free", getattr(self, f"{area}_free") - 1)
        events.append(f"patient_sent_to_{area}")

        if patient.patient_type == 0 and area == "resus":
            return 10.0
        if patient.patient_type == 1 and area == "major":
            return 6.0
        if patient.patient_type == 2 and area == "minor":
            return 3.0
        if patient.patient_type != 0 and area == "resus":
            return -5.0
        if patient.patient_type == 0 and area == "minor":
            return -8.0
        return 0.5

    def _request_imaging(self, events: List[str]) -> float:
        patient = self._pop_priority(self._treatment_patients)
        if patient is None:
            events.append("imaging_request_blocked")
            return -1.0
        patient.stage = 4
        patient.needs_imaging = True
        self._imaging_patients.append(patient)
        events.append("imaging_requested")
        return 1.0 if patient.patient_type in {0, 1} else 0.5

    def _process_imaging(self, events: List[str]) -> float:
        if not self._imaging_patients:
            events.append("process_imaging_blocked")
            return -1.0
        if self.radiologists_available <= 0:
            events.append("no_radiologist_available")
            return -1.0
        patient = self._pop_priority(self._imaging_patients)
        patient.needs_imaging = False
        patient.stage = 5
        self._treatment_patients.append(patient)
        events.append("imaging_processed")
        return 1.5

    def _treat_patient(self, events: List[str]) -> float:
        patient = self._pop_priority(self._treatment_patients)
        if patient is None:
            events.append("treatment_blocked")
            return -1.0
        if self.doctors_available <= 0:
            self._treatment_patients.appendleft(patient)
            events.append("no_doctor_available")
            return -1.0
        patient.stage = 5
        events.append("patient_treated")
        if patient.patient_type == 2:
            patient.stage = 11
            self._release_area(patient)
            events.append("minor_patient_ready_for_discharge")
            return 3.0
        self._admission_patients.append(patient)
        return 2.0

    def _admit_general(self, events: List[str]) -> float:
        patient = self._pop_priority(self._admission_patients)
        if patient is None:
            events.append("general_admission_blocked_no_patient")
            return -1.0
        if self.physicians_available <= 0 or self.general_free <= 0:
            self._admission_patients.appendleft(patient)
            events.append("general_admission_blocked")
            return -3.0
        patient.stage = 9
        self._release_area(patient)
        patient.area = "general"
        self._general_patients.append(patient)
        self.general_free -= 1
        events.append("patient_admitted_general")
        if patient.patient_type == 1:
            return 6.0
        if patient.patient_type == 2:
            return 3.0
        return 2.0

    def _admit_icu(self, events: List[str]) -> float:
        patient = self._pop_priority(self._admission_patients)
        if patient is None:
            events.append("icu_admission_blocked_no_patient")
            return -1.0
        if self.physicians_available <= 0 or self.icu_free <= 0:
            self._admission_patients.appendleft(patient)
            events.append("icu_admission_blocked")
            return -8.0 if patient.patient_type == 0 else -3.0
        patient.stage = 10
        self._release_area(patient)
        patient.area = "icu"
        self._icu_patients.append(patient)
        self.icu_free -= 1
        events.append("patient_admitted_icu")
        if patient.patient_type == 0:
            return 10.0
        return -5.0

    def _discharge_patient(self, events: List[str]) -> float:
        patient = self._pop_priority(self._admission_patients)
        if patient is None:
            patient = self._pop_priority(self._treatment_patients)
        if patient is None:
            patient = self._find_low_acuity_area_patient()
        if patient is None:
            events.append("discharge_blocked_no_patient")
            return -1.0
        patient.stage = 11
        self._release_area(patient)
        self._remove_from_all_active_queues(patient)
        events.append("patient_discharged")
        if patient.patient_type == 2:
            return 3.0 + 2.0
        return -6.0

    def _delay_patient(self, events: List[str]) -> float:
        patient = self._focus_patient()
        if patient is None:
            events.append("delay_no_patient")
            return -0.2
        patient.stage = 12
        patient.waiting_time += 1
        events.append("patient_delayed")
        if patient.patient_type == 0 and patient.waiting_time >= 4:
            return -10.0
        return -1.0

    def _preserve_icu_capacity(self, events: List[str]) -> float:
        self._preserve_icu_mode = True
        events.append("icu_capacity_preserved")
        if not self._has_emergency_waiting() and self.icu_free >= 1:
            return 1.0
        if self._has_emergency_waiting() and (self.icu_free > 0 or self.resus_free > 0):
            return -8.0
        return 0.0

    def _prioritise_emergency_flow(self, events: List[str]) -> float:
        for queue in self._front_door_queues():
            ordered = sorted(queue, key=lambda p: (p.patient_type, -p.waiting_time))
            queue.clear()
            queue.extend(ordered)
        events.append("emergency_flow_prioritised")
        return 2.0 if self._has_emergency_waiting() else 0.2

    def _release_capacity(self, events: List[str]) -> float:
        reward = 0.0
        reward += 0.5 * self._release_from_queue(self._resus_patients, "resus", self.resus_max, self.resus_release_probability, events)
        reward += 0.4 * self._release_from_queue(self._major_patients, "major", self.major_max, self.major_release_probability, events)
        reward += 0.3 * self._release_from_queue(self._minor_patients, "minor", self.minor_max, self.minor_release_probability, events)
        reward += 0.4 * self._release_from_queue(self._general_patients, "general", self.general_max, self.general_discharge_probability, events)
        reward += 0.5 * self._release_from_queue(self._icu_patients, "icu", self.icu_max, self.icu_discharge_probability, events)
        return reward

    def _release_from_queue(self, queue: Deque[Patient], area: str, capacity: int, probability: float, events: List[str]) -> int:
        kept = deque()
        released = 0
        while queue:
            patient = queue.popleft()
            if self._np_random.random() < probability:
                patient.stage = 11
                patient.area = None
                self._remove_from_all_active_queues(patient)
                released += 1
            else:
                kept.append(patient)
        queue.extend(kept)
        setattr(self, f"{area}_free", capacity - len(queue))
        if released:
            events.append(f"{area}_released_{released}")
        return released

    def _update_waiting_times(self) -> float:
        penalty = 0.0
        for queue in self._waiting_queues():
            for patient in queue:
                patient.waiting_time += 1
                penalty += 1.0
                if patient.patient_type == 0 and patient.waiting_time >= 4:
                    penalty += 10.0
        return penalty

    def _queue_congestion_penalty(self) -> float:
        congested = (
            len(self._registration_patients) >= 0.8 * self.registration_queue_max
            or len(self._assessment_patients) >= 0.8 * self.assessment_queue_max
        )
        return 3.0 if congested else 0.0

    def _icu_preservation_reward(self) -> float:
        if not self._has_emergency_waiting() and self.icu_free >= 1:
            return 1.0
        return 0.0

    def _new_patient(self) -> Patient:
        patient_type = int(self._np_random.choice([0, 1, 2], p=self.patient_type_probabilities))
        return Patient(patient_type=patient_type, stage=0)

    def _shift_from_timestep(self, timestep: int) -> str:
        hour = timestep % 24
        if 8 <= hour <= 15:
            return "day"
        if 16 <= hour <= 23:
            return "evening"
        return "night"

    def _apply_shift_staffing(self) -> None:
        staff = self.staff_schedule[self.shift]
        self.nurses_available = staff["nurses"]
        self.doctors_available = staff["doctors"]
        self.physicians_available = staff["physicians"]
        self.radiologists_available = staff["radiologists"]
        self.receptionists_available = staff["receptionists"]
        self.administrators_available = staff["administrators"]
        self.paramedics_available = staff["paramedics"]

    def _reset_public_state(self) -> None:
        self.timestep = 0
        self.shift = "night"
        self.registration_queue = 0
        self.assessment_queue = 0
        self.resus_free = self.resus_max
        self.major_free = self.major_max
        self.minor_free = self.minor_max
        self.general_free = self.general_max
        self.icu_free = self.icu_max
        self.patient_type = 2
        self.patient_stage = 0
        self.waiting_time = 0
        self._preserve_icu_mode = False
        self._apply_shift_staffing()

    def _sync_public_state(self) -> None:
        self.registration_queue = len(self._registration_patients)
        self.assessment_queue = len(self._assessment_patients)
        patient = self._focus_patient()
        self.patient_type = patient.patient_type if patient else 2
        self.patient_stage = patient.stage if patient else 11
        self.waiting_time = patient.waiting_time if patient else 0

    def _get_observation(self) -> np.ndarray:
        return np.asarray(
            [
                self.registration_queue,
                self.assessment_queue,
                self.resus_free,
                self.major_free,
                self.minor_free,
                self.general_free,
                self.icu_free,
                self.patient_type,
                self.patient_stage,
                self.waiting_time,
                self.nurses_available,
                self.doctors_available,
                self.physicians_available,
                self.radiologists_available,
                self.receptionists_available,
                self.administrators_available,
                self.paramedics_available,
                SHIFT_ENCODING[self.shift],
            ],
            dtype=np.float32,
        )

    def _get_info(self, action: Optional[int] = None, action_meaning: Optional[str] = None, events: Optional[List[str]] = None, reward_components: Optional[Dict[str, float]] = None) -> Dict[str, Any]:
        return {
            "timestep": self.timestep,
            "shift": self.shift,
            "action": action,
            "action_meaning": action_meaning,
            "events": events or [],
            "reward_components": reward_components or {},
            "patient_type_meaning": PATIENT_TYPES[self.patient_type],
            "patient_stage_meaning": PATIENT_STAGES[self.patient_stage],
            "queue_lengths": {
                "arrival": len(self._arrival_queue),
                "registration": len(self._registration_patients),
                "assessment": len(self._assessment_patients),
                "imaging": len(self._imaging_patients),
                "treatment": len(self._treatment_patients),
                "admission": len(self._admission_patients),
            },
            "free_capacity": {
                "resus": self.resus_free,
                "major": self.major_free,
                "minor": self.minor_free,
                "general": self.general_free,
                "icu": self.icu_free,
            },
        }

    def _pop_priority(self, queue: Deque[Patient]) -> Optional[Patient]:
        if not queue:
            return None
        best = min(queue, key=lambda p: (p.patient_type, -p.waiting_time))
        queue.remove(best)
        return best

    def _peek_priority(self, queue: Deque[Patient]) -> Optional[Patient]:
        if not queue:
            return None
        return min(queue, key=lambda p: (p.patient_type, -p.waiting_time))

    def _focus_patient(self) -> Optional[Patient]:
        candidates = []
        for queue in self._waiting_queues():
            candidates.extend(list(queue))
        if not candidates:
            return None
        return min(candidates, key=lambda p: (p.patient_type, -p.waiting_time))

    def _has_waiting_patients(self) -> bool:
        return any(len(queue) > 0 for queue in self._waiting_queues())

    def _has_emergency_waiting(self) -> bool:
        return any(patient.patient_type == 0 for queue in self._waiting_queues() for patient in queue)

    def _find_low_acuity_area_patient(self) -> Optional[Patient]:
        for queue in [self._minor_patients, self._major_patients, self._general_patients]:
            for patient in list(queue):
                if patient.patient_type == 2:
                    queue.remove(patient)
                    return patient
        return None

    def _release_area(self, patient: Patient) -> None:
        if patient.area not in {"resus", "major", "minor", "general", "icu"}:
            return
        area = patient.area
        queue = getattr(self, f"_{area}_patients")
        try:
            queue.remove(patient)
        except ValueError:
            pass
        free_attr = f"{area}_free"
        max_attr = f"{area}_max"
        setattr(self, free_attr, min(getattr(self, free_attr) + 1, getattr(self, max_attr)))
        patient.area = None

    def _remove_from_all_active_queues(self, patient: Patient) -> None:
        for queue in self._waiting_queues() + [
            self._resus_patients,
            self._major_patients,
            self._minor_patients,
            self._general_patients,
            self._icu_patients,
        ]:
            try:
                queue.remove(patient)
            except ValueError:
                pass

    def _front_door_queues(self) -> List[Deque[Patient]]:
        return [self._arrival_queue, self._registration_patients, self._assessment_patients]

    def _waiting_queues(self) -> List[Deque[Patient]]:
        return [
            self._arrival_queue,
            self._registration_patients,
            self._assessment_patients,
            self._imaging_patients,
            self._treatment_patients,
            self._admission_patients,
        ]

    def _all_queues(self) -> List[Deque[Patient]]:
        return self._waiting_queues() + [
            self._resus_patients,
            self._major_patients,
            self._minor_patients,
            self._general_patients,
            self._icu_patients,
        ]


## Random-Action Smoke Test

This test samples actions from `spaces.Discrete(15)` for 10 timesteps and prints the Gymnasium-style outputs. It does not train PPO.


In [15]:
env = HospitalPPOEnv(max_steps=10)
observation, info = env.reset(seed=42)

print("Initial observation:", observation)
print("Initial info:", info)

for step in range(10):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    print(f"Step {step + 1}")
    print("  action:", action, ACTION_MEANINGS[action])
    print("  observation:", observation)
    print("  reward:", reward)
    print("  terminated:", terminated)
    print("  truncated:", truncated)
    print("  info:", info)
    if terminated or truncated:
        break


Initial observation: [ 0.  0.  3.  8. 10. 20.  6.  2.  0.  0.  2.  1.  1.  1.  1.  1.  2.  2.]
Initial info: {'timestep': 0, 'shift': 'night', 'action': None, 'action_meaning': None, 'events': ['reset'], 'reward_components': {}, 'patient_type_meaning': 'elective_minor', 'patient_stage_meaning': 'arrival', 'queue_lengths': {'arrival': 1, 'registration': 0, 'assessment': 0, 'imaging': 0, 'treatment': 0, 'admission': 0}, 'free_capacity': {'resus': 3, 'major': 8, 'minor': 10, 'general': 20, 'icu': 6}}
Step 1
  action: 12 delay_patient
  observation: [ 0.  0.  3.  8. 10. 20.  6.  2. 12.  2.  2.  1.  1.  1.  1.  1.  2.  2.]
  reward: -2.0
  terminated: False
  truncated: False
  info: {'timestep': 1, 'shift': 'night', 'action': 12, 'action_meaning': 'delay_patient', 'events': ['new_patient_arrived', 'patient_delayed'], 'reward_components': {'release_throughput': 0.0, 'action_reward': -1.0, 'waiting_penalty': -2.0, 'queue_congestion_penalty': -0.0, 'icu_preservation_reward': 1.0}, 'patient_ty

## PPO Rollout Notation
For PPO, the environment supplies the state $s_t$, accepts the action $a_t$, and returns reward plus termination signals. These values form the rollout sequence used later for policy and value-function updates.


## Later PPO Connection

A future PPO implementation can use `env.observation_space.shape[0]` for the policy input and `env.action_space.n` for the 15 discrete actions. The CUDA guard means training code should place tensors/models on `DEVICE`, and the notebook will stop before rollout if CUDA is unavailable.
